# Implementation audit

Manuscript reference: Section VI.

The full audit covers $3679$ charge-neutral word observables across $26$ fixed-$N$ states with $n \in \{4, 6, 8\}$ and word lengths $m \in \{3, 4, 5\}$. The full re-run does not fit in free-tier Binder. This notebook does two things:

1. Loads the deposited screen JSONs and verifies the headline counts and the empirical effective constant $B^{\mathrm{eff}}_{\max} \approx 2.0$.
2. Runs a representative smoke audit on a single $n = 4$ Slater-determinant cell and confirms the bound is respected on every catalog observable.

**What this notebook verifies vs. does not verify.** Cells that load deposited JSON files and read out headline numbers (Sections "Headline counts" and "$B^{\mathrm{eff}}_{\max}$") verify the **archive**, not the underlying computation: they confirm the values in `data/*.json` match what the manuscript reports. The **science** (that those values are correct outputs of the pipeline) is verified by:

- `tests/test_smoke_regeneration.py`, which regenerates the Hubbard $U=0$ baseline from code and compares to the deposited file bit-exactly modulo volatile metadata;
- the "Smoke audit on a tiny cell" section below, which runs the catalog evaluation pipeline live on a single $n=4$ cell;
- the full audit, which is deterministically regenerable from code locally but exceeds Binder's free-tier memory.

Together with the SHA256 checks in `tests/test_data_integrity.py`, the archive-readback + smoke-regeneration pair gives end-to-end verification.

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"

for fn in ["screen_sector_cumulant_theorem.json",
          "screen_sector_cumulant_extended.json",
          "screen_sector_audit_r5.json",
          "audit_sector_cumulant_results.json"]:
    p = DATA / fn
    print(f"  {fn}: {'exists' if p.exists() else 'missing'}  ({p.stat().st_size if p.exists() else 0} bytes)")

  screen_sector_cumulant_theorem.json: exists  (3188 bytes)
  screen_sector_cumulant_extended.json: exists  (5613 bytes)
  screen_sector_audit_r5.json: exists  (4857 bytes)
  audit_sector_cumulant_results.json: exists  (2698 bytes)


## Headline counts from the deposited JSONs

Manuscript Sec VI: $3679$ observables = $630 + 1960 + 1089$ across the three audit screens. $26$ fixed-$N$ states = $6 + 11 + 9$.

In [2]:
screens = [
    "screen_sector_cumulant_theorem.json",
    "screen_sector_cumulant_extended.json",
    "screen_sector_audit_r5.json",
]

total_obs = 0
total_cells = 0
for fn in screens:
    p = DATA / fn
    if not p.exists():
        print(f"  {fn}: missing")
        continue
    j = json.loads(p.read_text(encoding="utf-8"))
    # Heuristic: the screen JSONs include a list of cells under one of
    # several common keys; sum their observable counts.
    cells = j.get("cells") or j.get("rows") or []
    n_obs = sum(c.get("n_obs", 0) for c in cells)
    print(f"  {fn}: {len(cells)} cells, {n_obs} observables")
    total_obs += n_obs
    total_cells += len(cells)

print()
print(f"Total observables across screens: {total_obs}  (manuscript: 3679)")
print(f"Total fixed-N cells:              {total_cells}  (manuscript: 26)")

  screen_sector_cumulant_theorem.json: 6 cells, 630 observables
  screen_sector_cumulant_extended.json: 11 cells, 1960 observables
  screen_sector_audit_r5.json: 9 cells, 1089 observables

Total observables across screens: 3679  (manuscript: 3679)
Total fixed-N cells:              26  (manuscript: 26)


## $B^{\mathrm{eff}}_{\max}$ on the audit suite

Manuscript: $B^{\mathrm{eff}}_{\max} \approx 2.0$ on the audit, indicating the universal $B_5 = 227\,251$ is empirically loose by ~$10^5$ on this catalog/state suite.

In [3]:
p = DATA / "screen_sector_audit_r5.json"
if p.exists():
    j = json.loads(p.read_text(encoding="utf-8"))
    # Try common locations for the headline value.
    val = (
        j.get("overall_B_eff_max")
        or j.get("B_eff_max")
        or j.get("summary", {}).get("B_eff_max")
    )
    if val is None:
        # Fallback: scan cells.
        cells = j.get("cells") or j.get("rows") or []
        candidates = [
            c.get("B_eff_max") for c in cells if c.get("B_eff_max") is not None
        ]
        val = max(candidates) if candidates else None
    print(f"B_eff_max from screen_sector_audit_r5.json: {val}")
    if val is not None:
        assert 1.5 < float(val) < 3.0, (
            f"B_eff_max out of expected ~2 range: {val}"
        )
        print("PASS: B_eff_max in [1.5, 3.0] (manuscript: ~2.0)")
else:
    print("audit JSON missing; skipping B_eff_max headline check")

B_eff_max from screen_sector_audit_r5.json: 2.0000125418392405
PASS: B_eff_max in [1.5, 3.0] (manuscript: ~2.0)


## Smoke audit on a tiny cell

Run the catalog evaluation pipeline on a single $n = 4$ Slater determinant. Section V proves $\Delta^{\mathrm{cat}}_{4,U(1)} = 0$ on every Slater determinant in the occupation basis, so we expect numerical zero for both $\Delta^{\mathrm{cat}}$ and $\max_W |\tau^G_W|$.

In [4]:
from connected_layer_sector import determinant_state, evaluate_catalog

n_orb = 4
rho = determinant_state(n_orb, [1, 1, 0, 0])
metrics = evaluate_catalog(rho, n_orb, r=4)
for k, v in metrics.items():
    print(f"  {k}: {v}")

# Sec V baseline: every cumulant of length 3 or 4 vanishes on a Slater
# determinant in the occupation basis.
assert metrics["delta_cat"] < 1e-12
assert metrics["max_tau"] < 1e-12
print()
print("PASS: Slater-determinant smoke audit reproduces Sec V zero baseline.")

  delta_cat: 0.0
  delta_cat_word: None
  max_tau: 0.0
  max_tau_word: None
  B_eff_max: 0.0
  eta_universal: nan
  B_r_universal: 105
  M_r_universal: 26
  n_catalog_words: 35

PASS: Slater-determinant smoke audit reproduces Sec V zero baseline.


## What's NOT here

The full $3679$-observable audit at $n = 8$ does not fit in Binder's free-tier memory. The deposited JSONs in `data/` carry the headline numbers; `tests/test_data_integrity.py` verifies their SHA256 against `MANIFEST.json`. To re-run the full audit locally, see the original screens in the project working directory.